In [30]:

from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [31]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [32]:
class CombinedModeWeightNet(nn.Module):
    def __init__(self, mode_model, weight_model):
        super().__init__()
        self.mode_model = mode_model
        self.weight_model = weight_model

    def forward(self, x_img, x_cond):
        mode = self.mode_model(x_img, x_cond)   # [B, 1]
        weight = self.weight_model(x_img, x_cond)  # [B, 1]
        return torch.cat((mode, weight), dim=1)   # [B, 2]

from only_mode_only_weight_v3 import No_normal_modewieght_net

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode0_model = No_normal_modewieght_net().to(device)
weight0_model = No_normal_modewieght_net().to(device)

mode0_model.load_state_dict(torch.load('models/blurred_first_mode.pth', map_location=torch.device(device)))
weight0_model.load_state_dict(torch.load('models/blurred_first_weight.pth', map_location=torch.device(device)))

cnn = CombinedModeWeightNet(mode0_model, weight0_model).to(device)
cnn.eval()

CombinedModeWeightNet(
  (mode_model): No_normal_modewieght_net(
    (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn2): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn3): GroupNorm(8, 256, eps=1e-05, affine=True)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv4): Conv2d(260, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn4): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn5): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv6): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (gn6): GroupNorm(8, 256, eps=1e-05, affine=True)
    (conv7): Conv2d(256, 256, kernel_size=(3, 3),

In [33]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        '''
        standard ResNet style convolutional block, for image processing
        '''
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            # this adds on correct residual in case channels have increased
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2


In [34]:
class UnetDown(nn.Module):
    """
    Downsampling path for U-Net, reduces spatial resolution while increasing feature depth
    Input: Image batch, size (batchsize, 1, 32, 32)
    Output: size (batchsize, out_channels, 16, 16)
    Output:
    """
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        '''
        process and downscale the image feature maps
        '''
        layers = [ResidualConvBlock(
            in_channels, out_channels), nn.MaxPool2d(2)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Doubles spatial dimensions, halves feature dimensions
        # My channel dimension for image will always be 1, greyscale
        return self.model(x)


In [35]:
class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        '''
        process and upscale the image feature maps
        Doubles spatial size, but decreases channels:
        input: 2 vectors of size (binsize, in_channels / 2, h, w) 
        output: (binsize, outchannels, 2h, 2w)
        '''
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        """
        x is the upsampled features from previous decoder layer
        skip is the skip connection from the encoder, same size as x
        """
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x

In [36]:
class EmbedFC(nn.Module):
    """
    Use FC layer for embedding 1-d metadata, like modes+weights
    (putting into higher dimension)
    Effectively our conditional
    input: Conditional, size (batchsize, input_dim = 4+4)
    Output: Higherdimensional tensor, size (batchsize, output_dim)
    
    """
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things  
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assumes you already have these:
# - ResidualConvBlock(in_ch, out_ch, is_res=True)
# - UnetDown(in_ch, out_ch)
# - UnetUp(in_ch, out_ch)
# - EmbedFC(in_dim, out_dim)

class ContextUnet(nn.Module):
    """
    U-Net for conditional image generation (32x32), where conditioning is
    provided by *tiled* scalar maps concatenated to the input channels.

    Conditioning (this version):
      - top (mode, weight): shape [B, 2]  (or [B, 1, 2], both supported)
      - params:             shape [B, 4]
      -> total 6 scalars per sample, each tiled to [H, W] and concatenated as channels.

    Timestep t is still embedded via an MLP and injected on the up path.
    """

    def __init__(self, in_channels=1, n_feat=256, use_time_embed=True):
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.use_time_embed = use_time_embed

        # 6 tiled condition channels (2 from top mode/weight + 4 params)
        self.cond_channels = 6
        lifted_in = in_channels + self.cond_channels

        # Encoder
        self.init_conv = ResidualConvBlock(lifted_in, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)          # 32x32 -> 16x16 (n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)      # 16x16 -> 8x8   (2*n_feat)

        # Latent pooling to 1x1
        self.to_vec = nn.Sequential(nn.AvgPool2d(8), nn.GELU())

        # Optional timestep embedding (kept as MLP, not tiled)
        if use_time_embed:
            self.timeembed1 = EmbedFC(1, 2 * n_feat)
            self.timeembed2 = EmbedFC(1, 1 * n_feat)

        # Decoder
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8),  # 1x1 -> 8x8
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        # Concats with skips: (2*n_feat up) + (2*n_feat skip) = 4*n_feat -> n_feat
        self.up1 = UnetUp(4 * n_feat, n_feat)          # 8x8 -> 16x16
        # (n_feat up) + (n_feat skip) = 2*n_feat -> n_feat
        self.up2 = UnetUp(2 * n_feat, n_feat)          # 16x16 -> 32x32

        # Output head: concat with early skip (x after init_conv) -> 2*n_feat
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    @staticmethod
    def _tile_condition(top_pair, params, H, W):
        """
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        Returns tiled tensor of shape [B, 6, H, W]
        """
        if top_pair.dim() == 3:
            # allow [B, 1, 2] → [B, 2]
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2] (mode, weight)"

        B = top_pair.shape[0]
        cond = torch.cat([top_pair, params], dim=1)    # [B, 6]
        cond = cond.unsqueeze(-1).unsqueeze(-1)        # [B, 6, 1, 1]
        cond = cond.repeat(1, 1, H, W)                 # [B, 6, H, W]
        return cond

    def forward(self, x, top_pair, params, t):
        """
        x:        [B, 1, 32, 32]  (noisy waveguide / latent)
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        t:        [B, 1] (scalar timestep per sample)
        """
        B, _, H, W = x.shape
        assert H == 32 and W == 32, "This U-Net assumes 32x32 spatial size."

        # Build and add tiled conditions as channels at the input
        cond_maps = self._tile_condition(top_pair, params, H, W)  # [B, 6, 32, 32]
        x_in = torch.cat([x, cond_maps], dim=1)                   # [B, 1+6, 32, 32]

        # Encoder
        x0 = self.init_conv(x_in)   # [B, n_feat, 32, 32]
        d1 = self.down1(x0)         # [B, n_feat, 16, 16]
        d2 = self.down2(d1)         # [B, 2*n_feat, 8, 8]
        h  = self.to_vec(d2)        # [B, 2*n_feat, 1, 1]

        # Decode (optionally inject time embeddings)
        up1 = self.up0(h)           # [B, 2*n_feat, 8, 8]

        if self.use_time_embed:
            temb1 = self.timeembed1(t).view(B, 2 * self.n_feat, 1, 1)
            temb2 = self.timeembed2(t).view(B, 1 * self.n_feat, 1, 1)
            up1 = up1 + temb1

        u2 = self.up1(up1, d2)      # -> [B, n_feat, 16, 16]
        if self.use_time_embed:
            u2 = u2 + temb2

        u3 = self.up2(u2, d1)       # -> [B, n_feat, 32, 32]

        # Final head with early skip (x0)
        out = self.out(torch.cat([u3, x0], dim=1))  # [B, 1, 32, 32]
        return out


In [38]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [39]:
import torch
import torch.nn as nn
import numpy as np

class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model = nn_model.to(device)

        # register all schedule buffers
        sched = ddpm_schedules(betas[0], betas[1], n_T)
        for k, v in sched.items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    @staticmethod
    def _apply_mask(top_pair, params, context_mask):
        """
        context_mask: [B] or [B,1] of {0,1}, where 1 => drop conditioning.
        Returns masked copies (zeros when dropped).

        top_pair: [B, 2] or [B, 1, 2]
        params:   [B, 4]
        """
        if context_mask.dim() == 1:
            context_mask = context_mask.unsqueeze(1)  # [B,1]

        # Normalize top_pair shape to [B, 2]
        if top_pair.dim() == 3:
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2]"

        tp = top_pair * (1.0 - context_mask)            # [B,2] broadcast
        pr = params   * (1.0 - context_mask)            # [B,4] broadcast
        return tp, pr

    def forward(self, x, top_pair, params):
        """
        Training step:
          x:        [B,1,32,32] (clean image x0)
          top_pair: [B,2] (mode_0, weight_0)  or [B,1,2]
          params:   [B,4]
        """
        B = x.shape[0]
        _ts = torch.randint(1, self.n_T + 1, (B,), device=self.device)        # [B]
        noise = torch.randn_like(x)                                           # [B,1,32,32]

        x_t = (
            self.sqrtab[_ts, None, None, None] * x
            + self.sqrtmab[_ts, None, None, None] * noise
        )

        # classifier-free dropout
        context_mask = torch.bernoulli(
            torch.full((B,), self.drop_prob, device=self.device)
        )  # [B] in {0,1}

        tp_masked, pr_masked = self._apply_mask(top_pair, params, context_mask)

        # normalized timestep as [B,1]
        t_norm = (_ts.float() / self.n_T).unsqueeze(1)  # [B,1]

        pred_noise = self.nn_model(x_t, tp_masked, pr_masked, t_norm)
        return self.loss_mse(noise, pred_noise)

    @torch.no_grad()
    def sample(self, n_sample, size, device, top_pair, params, guide_w=0.0):
        """
        Sampling with classifier-free guidance (CFG).

        Args
        ----
        n_sample: int
        size:     tuple like (1, 32, 32)
        device:   torch.device
        top_pair: [n_sample, 2] tensor (mode_0, weight_0)  or [n_sample, 1, 2]
        params:   [n_sample, 4] tensor
        guide_w:  float guidance scale (0 = no CFG)

        Returns
        -------
        x_T->x_0 sample tensor [n_sample, 1, 32, 32], and numpy trajectory.
        """
        assert top_pair.shape[0] == n_sample and params.shape[0] == n_sample

        x_i = torch.randn(n_sample, *size, device=device)

        # Build masks for double batch (first half conditioned, second half dropped)
        context_mask_cond   = torch.zeros(n_sample, device=device)  # keep
        context_mask_uncond = torch.ones(n_sample,  device=device)  # drop

        # Precompute cond/uncond views
        tp_cond, pr_cond       = self._apply_mask(top_pair, params, context_mask_cond)
        tp_uncond, pr_uncond   = self._apply_mask(top_pair, params, context_mask_uncond)

        x_i_store = []
        for i in range(self.n_T, 0, -1):
            # timestep scalar normalized
            t_norm = torch.full((n_sample, 1), i / self.n_T, device=device)

            # double the batch (conditioned + unconditioned)
            x_in = torch.cat([x_i, x_i], dim=0)
            t_in = torch.cat([t_norm, t_norm], dim=0)             # [2B,1]
            tp_in = torch.cat([tp_cond, tp_uncond], dim=0)        # [2B,2]
            pr_in = torch.cat([pr_cond, pr_uncond], dim=0)        # [2B,4]

            # predict noise for both halves
            eps = self.nn_model(x_in, tp_in, pr_in, t_in)         # [2B,1,32,32]
            eps1, eps2 = eps[:n_sample], eps[n_sample:]           # cond, uncond

            # CFG combine
            eps_cfg = (1 + guide_w) * eps1 - guide_w * eps2

            z = torch.randn_like(x_i) if i > 1 else 0.0
            x_i = (
                self.oneover_sqrta[i] * (x_i - eps_cfg * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )

            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        x_i_store = np.array(x_i_store)
        return x_i, x_i_store


In [40]:
import importlib
import waveguide_dataset_paired_nonorm
importlib.reload(waveguide_dataset_paired_nonorm)
from waveguide_dataset_paired_nonorm import WaveguideDatasetPaired

In [ ]:
def _to_top_pair(cond_or_top):
    """
    Accepts:
      - cond_or_top [B,4,2]: (modes, weights) → returns top pair [B,2] via [:,0,:]
      - cond_or_top [B,2]: already top pair
    """
    if cond_or_top.dim() == 3:
        assert cond_or_top.size(2) == 2 and cond_or_top.size(1) >= 1, \
            f"Expected [B,4,2]-like, got {tuple(cond_or_top.shape)}"
        return cond_or_top[:, 0, :]  # [B,2]
    assert cond_or_top.dim() == 2 and cond_or_top.size(1) == 2, \
        f"Expected [B,2], got {tuple(cond_or_top.shape)}"
    return cond_or_top

@torch.no_grad()
def compare_waveguides_3gens1(
    ddpm, test_loader, device, guide_w=2.0, n_rows=12,
    save_path=None, binarize=False, thresh=0.5
):
    """
    For each of the first n_rows items from the test loader, generate 3 waveguides
    conditioned on the same (top_pair, params), and display as:
        Real (red) | Gen 1 (black) | Gen 2 (black) | Gen 3 (black)

    Expects test_loader to yield either:
      (cond:[B,4,2], params:[B,4], x_real:[B,1,32,32])  OR
      (top_pair:[B,2], params:[B,4], x_real:[B,1,32,32])
    """
    ddpm.eval()

    # Grab one test batch
    try:
        cond_or_top, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    n = min(n_rows, cond_or_top.shape[0])
    cond_or_top = cond_or_top[:n].to(device)
    params      = params[:n].to(device)
    x_real      = x_real[:n].to(device)

    # Convert to [n,2] top pair
    top_pair = _to_top_pair(cond_or_top)  # [n,2]

    # Repeat each condition 3× to get 3 generated samples per item
    num_gens     = 3
    top_pair_rep = top_pair.repeat_interleave(num_gens, dim=0)   # [n*3,2]
    params_rep   = params.repeat_interleave(num_gens, dim=0)     # [n*3,4]

    # Sample all at once (DDPM randomness gives different outputs per repeat)
    x_gen_all, _ = ddpm.sample(
        n_sample=n * num_gens,
        size=(1, 32, 32),
        device=device,
        top_pair=top_pair_rep,   # <-- updated
        params=params_rep,
        guide_w=guide_w,
    )  # [n*3,1,32,32]

    # Reshape to [n, 3, 1, 32, 32]
    x_gen_all = x_gen_all.view(n, num_gens, 1, 32, 32)

    # To CPU numpy
    real_np = x_real.detach().cpu().numpy()           # [n,1,32,32]
    gen_np  = x_gen_all.detach().cpu().numpy()        # [n,3,1,32,32]

    # Clamp and optional binarization
    real_np = np.clip(real_np, 0.0, 1.0)
    gen_np  = np.clip(gen_np,  0.0, 1.0)
    if binarize:
        real_np = (real_np >= thresh).astype(np.float32)
        gen_np  = (gen_np  >= thresh).astype(np.float32)

    # Plot: 4 columns => Real + 3 gens
    cols = 4
    fig_h = max(2, n * 1.1)
    fig, axs = plt.subplots(n, cols, figsize=(cols * 2.2, fig_h), squeeze=False)

    for i in range(n):
        # Real (red)
        ax = axs[i, 0]
        ax.imshow(real_np[i, 0], cmap="Reds", vmin=0, vmax=1)
        if i == 0: ax.set_title("Real", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # Gen 1..3 (black). Use 1 - gen for black-on-white with 'binary' cmap
        for j in range(num_gens):
            ax = axs[i, j + 1]
            ax.imshow(1.0 - gen_np[i, j, 0], cmap="binary_r", vmin=0, vmax=1)
            if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()


In [48]:
import torch
import numpy as np
import matplotlib.pyplot as plt

@torch.no_grad()
def compare_waveguides_3gens_with_cnn(
    ddpm,
    cnn,
    test_loader,
    device,
    guide_w: float = 2.0,
    n_rows: int = 12,
    save_path: str | None = None,
):
    """
    For each of the first n_rows items from the test loader:
      - Generate 3 waveguides conditioned on the same (top_pair, params)
      - Show 4 images (Real | Gen1 | Gen2 | Gen3), with per-row shared vmin/vmax (no clamping)
      - Show 2 small plots (Top Mode, Top Weight) with CNN outputs:
            green = each generated sample, red = target (plotted last)

    test_loader must yield either:
      (cond:[B,4,2], params:[B,4], x_real:[B,1,H,W])  OR
      (top_pair:[B,2], params:[B,4], x_real:[B,1,H,W])
    """
    ddpm.eval()
    cnn.eval()

    # ---- get one batch
    try:
        cond_or_top, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    n = min(n_rows, cond_or_top.shape[0])
    cond_or_top = cond_or_top[:n].to(device)
    params      = params[:n].to(device)
    x_real      = x_real[:n].to(device)  # NOT clamped

    # ---- handle cond format → top_pair:[n,2]
    def _to_top_pair(t: torch.Tensor) -> torch.Tensor:
        if t.dim() == 2 and t.size(1) == 2:                 # [n,2]
            return t
        if t.dim() == 3 and t.size(1) == 4 and t.size(2) == 2:  # [n,4,2]
            return t[:, 0, :]  # use the first pair as "top"
        raise ValueError(f"Unrecognized cond/top shape: {tuple(t.shape)}")

    top_pair = _to_top_pair(cond_or_top)  # [n,2]

    # ---- generate 3 samples per item
    num_gens = 3
    top_pair_rep = top_pair.repeat_interleave(num_gens, dim=0)   # [n*3,2]
    params_rep   = params.repeat_interleave(num_gens, dim=0)     # [n*3,4]

    # DDPM sampling (outputs NOT clamped)
    B, _, H, W = x_real.shape
    x_gen_all, _ = ddpm.sample(
        n_sample=n * num_gens,
        size=(1, H, W),
        device=device,
        top_pair=top_pair_rep,
        params=params_rep,
        guide_w=guide_w,
    )  # [n*3,1,H,W]
    x_gen_all = x_gen_all.view(n, num_gens, 1, H, W)   # [n,3,1,H,W]

    # ---- CNN predictions (target + gens)  (inputs NOT clamped)
    def _extract_mode_weight(y: torch.Tensor):
        D = y.shape[-1]
        if D == 2:
            return y[..., 0], y[..., 1]
        if D >= 5:
            return y[..., 0], y[..., 4]
        raise ValueError(f"CNN output last-dim {D} not supported (expect 2 or ≥5).")

    y_real = cnn(x_real, params)                    # [n, D]
    mode_real, wt_real = _extract_mode_weight(y_real)

    y_gen  = cnn(x_gen_all.reshape(n * num_gens, 1, H, W), params_rep)  # [n*3, D]
    mode_gen, wt_gen = _extract_mode_weight(y_gen)
    mode_gen = mode_gen.view(n, num_gens)
    wt_gen   = wt_gen.view(n, num_gens)

    # ---- to CPU numpy for plotting (NO clamping)
    real_np = x_real.detach().cpu().numpy()            # [n,1,H,W]
    gen_np  = x_gen_all.detach().cpu().numpy()         # [n,3,1,H,W]
    mode_real_np = mode_real.detach().cpu().numpy()
    wt_real_np   = wt_real.detach().cpu().numpy()
    mode_gen_np  = mode_gen.detach().cpu().numpy()
    wt_gen_np    = wt_gen.detach().cpu().numpy()

    # ---- figure layout: 6 columns (4 images + 2 small plots)
    cols = 6
    fig_h = max(2.0, n * 1.3)
    fig_w = 2.2 * 4 + 1.8 * 2
    fig, axs = plt.subplots(
        n, cols, figsize=(fig_w, fig_h),
        gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.8, 0.8]},
        squeeze=False
    )

    for i in range(n):
        # Shared vmin/vmax across the 4 images in this row (no clamping)
        row_min = np.nanmin([
            real_np[i, 0].min(),
            gen_np[i, 0, 0].min(),
            gen_np[i, 1, 0].min(),
            gen_np[i, 2, 0].min(),
        ])
        row_max = np.nanmax([
            real_np[i, 0].max(),
            gen_np[i, 0, 0].max(),
            gen_np[i, 1, 0].max(),
            gen_np[i, 2, 0].max(),
        ])

        # Real
        ax = axs[i, 0]
        ax.imshow(real_np[i, 0], cmap="Reds", vmin=row_min, vmax=row_max)
        if i == 0: ax.set_title("Real", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # Gens 1..3
        for j in range(num_gens):
            ax = axs[i, j + 1]
            ax.imshow(gen_np[i, j, 0], cmap="gray", vmin=row_min, vmax=row_max)
            if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

        # Small plots: Top Mode (col 4) and Top Weight (col 5)
        xs = np.array([1, 2, 3, 4])

        axm = axs[i, 4]
        axm.scatter([1, 2, 3], mode_gen_np[i], c="green", s=15, label="Gen")
        axm.scatter([4], [mode_real_np[i]], c="red", s=20, label="Target")  # plotted last
        if i == 0:
            axm.set_title("Top Mode (CNN)", fontsize=10)
            axm.legend(loc="best", fontsize=7, frameon=False)
        axm.set_xticks(xs, ["G1", "G2", "G3", "T"])
        axm.tick_params(axis='both', labelsize=8)
        axm.grid(alpha=0.2, linewidth=0.6)

        axw = axs[i, 5]
        axw.scatter([1, 2, 3], wt_gen_np[i], c="green", s=15, label="Gen")
        axw.scatter([4], [wt_real_np[i]], c="red", s=20, label="Target")   # plotted last
        if i == 0:
            axw.set_title("Top Weight (CNN)", fontsize=10)
        axw.set_xticks(xs, ["G1", "G2", "G3", "T"])
        axw.tick_params(axis='both', labelsize=8)
        axw.grid(alpha=0.2, linewidth=0.6)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def train_waveguide_ddpm(
    h5_path='train_test_split_soft_fp16.h5',
    stats_path="waveguide_stats_log_norm_above90.npz",
    save_dir="./data/blurred_diffusion_v1/",
    n_epoch=10,
    batch_size=256,
    n_T=400,
    n_feat=128,
    lrate=1e-4,
    drop_prob=0.1,
    betas=(1e-4, 0.02),
    ws_test=(0.0, 0.5, 2.0),
    save_model=True,
    test_eval_fraction=0.25,   # portion of test loader to evaluate each epoch
    device=None,
    cnn=None,                  # NEW: evaluator CNN (x:[B,1,32,32], params:[B,4]) -> [B,2]
    gamma: float = 0.0,        # NEW: blend: gamma*noise_loss + (1-gamma)*cnn_loss
):
    assert 0.0 <= gamma <= 1.0, "gamma must be in [0,1]"
    if gamma > 0 and cnn is None:
        raise ValueError("gamma>0 requires a CNN (cnn=...)")

    os.makedirs(save_dir, exist_ok=True)
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")

    # ---- Data ----
    train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
    test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # ---- Model ----
    unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
    ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)
    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    # ---- Helpers ----
    def extract_top_pair(cond_b42):
        """
        cond_b42: [B, 4, 2]  (modes, weights) in order of importance.
        Return top (mode, weight) as [B, 2].
        """
        assert cond_b42.dim() == 3 and cond_b42.size(1) >= 1 and cond_b42.size(2) == 2, \
            f"Expected [B,4,2]-like, got {tuple(cond_b42.shape)}"
        return cond_b42[:, 0, :]  # [B,2] → (top mode, top weight)

    # Prep CNN (fixed evaluator) if used
    use_cnn = (cnn is not None) and (gamma < 1.0)
    if use_cnn:
        cnn = cnn.to(device).eval()
        for p in cnn.parameters():
            p.requires_grad_(False)

    # ---- Tracking ----
    train_losses, test_losses = [], []
    best_test_loss = float("inf")

    def evaluate_on_test_portion():
        """Blended loss on a subset: gamma*noise + (1-gamma)*cnn, with dropout disabled."""
        ddpm.eval()
        was = ddpm.drop_prob
        ddpm.drop_prob = 0.0  # disable CFG dropout during eval

        total, count = 0.0, 0
        max_batches = max(1, int(np.ceil(test_eval_fraction * len(test_loader)))) if len(test_loader) > 0 else 1

        with torch.no_grad():
            for b_idx, (cond, params, x_real) in enumerate(test_loader):
                x_real = x_real.to(device, non_blocking=True)  # [B,1,32,32]
                cond   = cond.to(device, non_blocking=True)    # [B,4,2]
                params = params.to(device, non_blocking=True)  # [B,4]
                top_pair = extract_top_pair(cond)              # [B,2]

                # Noise loss via your existing API (unchanged)
                loss_noise = ddpm(x_real, top_pair, params)

                # CNN loss (eval path without dropout; single t/noise)
                if use_cnn and gamma < 1.0:
                    B = x_real.size(0)
                    t = torch.randint(1, n_T + 1, (B,), device=device)
                    noise = torch.randn_like(x_real)
                    x_t = ddpm.sqrtab[t, None, None, None] * x_real + ddpm.sqrtmab[t, None, None, None] * noise
                    t_norm = (t.float() / n_T).unsqueeze(1)

                    pred_noise = ddpm.nn_model(x_t, top_pair, params, t_norm)
                    x0_hat = (x_t - ddpm.sqrtmab[t, None, None, None] * pred_noise) / ddpm.sqrtab[t, None, None, None]
                    x0_hat = x0_hat.clamp(0.0, 1.0)

                    targ_top = cnn(x_real, params)  # [B,2] from target
                    pred_top = cnn(x0_hat, params)  # [B,2] from generated
                    loss_cnn = F.mse_loss(pred_top, targ_top)
                else:
                    loss_cnn = x_real.new_zeros(())

                loss = gamma * loss_noise + (1.0 - gamma) * loss_cnn
                total += float(loss.item())
                count += 1
                if b_idx + 1 >= max_batches:
                    break

        ddpm.drop_prob = was
        return total / max(count, 1)

    # ---- Training loop ----
    for ep in range(n_epoch):
        print(f"epoch {ep}")
        ddpm.train()

        # linear LR decay
        for g in optim.param_groups:
            g["lr"] = lrate * (1 - ep / n_epoch)

        pbar = tqdm(train_loader)
        loss_ema = None
        train_loss_sum, train_loss_batches = 0.0, 0

        for cond, params, x_real in pbar:
            x_real = x_real.to(device, non_blocking=True)  # [B,1,32,32]
            cond   = cond.to(device, non_blocking=True)    # [B,4,2]
            params = params.to(device, non_blocking=True)  # [B,4]
            top_pair = extract_top_pair(cond)              # [B,2]

            # 1) Noise loss via your existing API (unchanged)
            optim.zero_grad(set_to_none=True)
            loss_noise = ddpm(x_real, top_pair, params)

            # 2) CNN loss by comparing cnn(x_real, params) vs cnn(x0_hat, params)
            if use_cnn and gamma < 1.0:
                # build one-step reconstruction x0_hat using the same DDPM components
                B = x_real.size(0)
                t = torch.randint(1, n_T + 1, (B,), device=device)
                noise = torch.randn_like(x_real)
                x_t = ddpm.sqrtab[t, None, None, None] * x_real + ddpm.sqrtmab[t, None, None, None] * noise
                t_norm = (t.float() / n_T).unsqueeze(1)

                # classifier-free guidance dropout on conditioning (same drop_prob as model)
                context_mask = torch.bernoulli(torch.full((B,), ddpm.drop_prob, device=device))
                ctx = context_mask.unsqueeze(1)
                top_pair_m = top_pair * (1.0 - ctx)
                params_m   = params   * (1.0 - ctx)

                pred_noise = ddpm.nn_model(x_t, top_pair_m, params_m, t_norm)
                x0_hat = (x_t - ddpm.sqrtmab[t, None, None, None] * pred_noise) / ddpm.sqrtab[t, None, None, None]
                x0_hat = x0_hat.clamp(0.0, 1.0)

                with torch.no_grad():
                    targ_top = cnn(x_real, params)  # fixed target from real waveguide
                pred_top = cnn(x0_hat, params)

                loss_cnn = F.mse_loss(pred_top, targ_top)
            else:
                loss_cnn = x_real.new_zeros(())

            # 3) Blended loss and step
            loss = gamma * loss_noise + (1.0 - gamma) * loss_cnn
            loss.backward()
            optim.step()

            loss_val = float(loss.detach().item())
            train_loss_sum += loss_val
            train_loss_batches += 1

            # display
            l_noise = float(loss_noise.detach().item())
            l_cnn   = float(loss_cnn.detach().item()) if (use_cnn and gamma < 1.0) else 0.0
            loss_ema = loss_val if loss_ema is None else (0.95 * loss_ema + 0.05 * loss_val)
            pbar.set_description(f"loss:{loss_ema:.4f} | noise:{l_noise:.3e} | cnn:{l_cnn:.3e}")

        # Mean train loss this epoch
        mean_train_loss = train_loss_sum / max(train_loss_batches, 1)
        train_losses.append(mean_train_loss)

        # ---- Evaluation on test subset (blended) ----
        mean_test_loss = evaluate_on_test_portion()
        test_losses.append(mean_test_loss)
        print(f"epoch {ep}: train_loss={mean_train_loss:.6f} | test_loss={mean_test_loss:.6f}")

        # ---- Save best model ----
        if save_model and mean_test_loss < best_test_loss:
            best_test_loss = mean_test_loss
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(ddpm.state_dict(), best_path)
            print(f"✔ improved test loss; saved best model to {best_path}")


        # ---- Plot & save loss curves (updates every epoch) ----
        try:
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(train_losses)+1), train_losses, label="Train (blended)")
            plt.plot(range(1, len(test_losses)+1),  test_losses,  label="Test (blended)")
            plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"Train vs Test Loss (gamma={gamma})")
            plt.legend(); plt.tight_layout()
            loss_curve_path = os.path.join(save_dir, "loss_curve.png")
            plt.savefig(loss_curve_path, dpi=150)
            plt.close()
            print(f"updated loss curve at {loss_curve_path}")
        except Exception as e:
            print(f"warning: failed to plot loss curve ({e})")

    # Optionally save final model state
    if save_model:
        final_path = os.path.join(save_dir, f"model_final.pth")
        torch.save(ddpm.state_dict(), final_path)
        print(f"saved final model at {final_path}")
    
    # ---- FINAL-EPOCH SUMMARY IMAGE (replaces GIFs) ----
    # Uses your provided compare function; saves to save_dir/compare_3gens.png with 12 rows.
    try:
        out_png = os.path.join(save_dir, "compare_3gens.png")
        compare_waveguides_3gens(
            ddpm,
            test_loader,
            device=device,
            guide_w=2.0,
            n_rows=12,
            save_path=out_png,
            binarize=False,
            thresh=0.5,
        )
    except Exception as e:
        print(f"warning: failed to create final comparison grid ({e})")


In [43]:
if __name__ == "__main__":
    train_waveguide_ddpm(cnn=cnn, gamma=1, save_dir="./data/blurred_diffusion_gamma10/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.9, save_dir="./data/blurred_diffusion_gamma90/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.5, save_dir="./data/blurred_diffusion_gamma50/")

epoch 0


loss:0.0143 | noise:1.769e-02 | cnn:0.000e+00: 100%|██████████| 3501/3501 [05:58<00:00,  9.75it/s]


epoch 0: train_loss=0.040252 | test_loss=0.013242
✔ improved test loss; saved best model to ./data/blurred_diffusion_gamma10/best_model.pth
updated loss curve at ./data/blurred_diffusion_gamma10/loss_curve.png
epoch 1


loss:0.0089 | noise:8.859e-03 | cnn:0.000e+00: 100%|██████████| 3501/3501 [05:49<00:00, 10.00it/s]


epoch 1: train_loss=0.010908 | test_loss=0.008170
✔ improved test loss; saved best model to ./data/blurred_diffusion_gamma10/best_model.pth
updated loss curve at ./data/blurred_diffusion_gamma10/loss_curve.png
epoch 2


...

epoch 7: train_loss=2.966494 | test_loss=3.100098
✔ improved test loss; saved best model to ./data/blurred_diffusion_gamma50/best_model.pth
updated loss curve at ./data/blurred_diffusion_gamma50/loss_curve.png
epoch 8


loss:3.0515 | noise:1.797e-01 | cnn:3.597e+00: 100%|██████████| 3501/3501 [12:37<00:00,  4.62it/s]


epoch 8: train_loss=2.864039 | test_loss=2.929762
✔ improved test loss; saved best model to ./data/blurred_diffusion_gamma50/best_model.pth
updated loss curve at ./data/blurred_diffusion_gamma50/loss_curve.png
epoch 9


loss:2.5451 | noise:1.456e-01 | cnn:6.860e+00: 100%|██████████| 3501/3501 [12:38<00:00,  4.61it/s]


epoch 9: train_loss=2.832626 | test_loss=2.913968
✔ improved test loss; saved best model to ./data/blurred_diffusion_gamma50/best_model.pth
updated loss curve at ./data/blurred_diffusion_gamma50/loss_curve.png
saved final model at ./data/blurred_diffusion_gamma50/model_final.pth
Saved comparison grid to ./data/blurred_diffusion_gamma50/compare_3gens.png


In [57]:
# import torch
# import numpy as np
# import matplotlib.pyplot as plt



h5_path='train_test_split_soft_fp16.h5'
stats_path = "waveguide_stats_log_norm_above90.npz"
n_epoch=20
batch_size=256
n_T=400
n_feat=128
lrate=1e-4
drop_prob=0.1
betas=(1e-4, 0.02)
ws_test=(0.0, 0.5, 2.0)
save_model=True
test_eval_fraction=0.25   # portion of test loader to evaluate each epoch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ---- Data ----
train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# ---- Model ----
unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)
ddpm.load_state_dict(torch.load('data/blurred_diffusion_gamma10/best_model.pth', map_location=torch.device(device)))

compare_waveguides_3gens_with_cnn(ddpm,cnn, test_loader, device, guide_w=2.0, n_rows=12,
                          save_path="data/blurred_diffusion_gamma10/compare_3gens_wcnnn.png",
                          )


Saved comparison grid to data/blurred_diffusion_gamma10/compare_3gens_wcnnn.png
